<a href="https://colab.research.google.com/github/vasanthkumarreddy1/Telecustomer-churn-prediction-main/blob/main/Copy_of_Z_Image_With__Lora_Support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background-color: #161b22; padding: 20px; border-radius: 12px; border-left: 6px solid #2ecc71; box-shadow: 0 4px 6px rgba(0,0,0,0.3);">
  <h1 style="color: #ffffff; margin: 0; font-family: sans-serif; font-weight: 700; letter-spacing: 1px;">
    ⚡ Z-Image Turbo <span style="font-size: 0.6em; color: #2ecc71; vertical-align: middle; background: #2ecc7122; padding: 2px 8px; border-radius: 6px;">With Lora Support</span>
  </h1>
  <p style="color: #8b949e; margin: 8px 0 0 0; font-family: sans-serif;">
    • Hybrid FP8/GGUF + LoRA Optimized
  </p>
</div>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**
*Subscribe for more AI tutorials, workflows, and optimization tips!*

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the ephemeral storage, installs ComfyUI, and configures the GGUF integration tools.

import os
import subprocess

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 Installing Dependencies (This takes a moment)...")
!cd {LOCAL_WORKSPACE} && pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 &> /dev/null

GGUF_NODE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-GGUF")
if not os.path.exists(GGUF_NODE_DIR):
    print("🧩 Installing GGUF Processing Nodes...")
    !git clone https://github.com/city96/ComfyUI-GGUF {GGUF_NODE_DIR} &> /dev/null
    !pip install -r {GGUF_NODE_DIR}/requirements.txt &> /dev/null

print("✅ Environment Ready!")

🚀 Initializing Core Architecture...
   ✓ Core Engine Cloned
📦 Installing Dependencies (This takes a moment)...
🧩 Installing GGUF Processing Nodes...
✅ Environment Ready!


In [ ]:

# @title 2. High-Speed Asset Downloader (HuggingFace + Authenticated Civitai)
# @markdown Downloads Z-Image Turbo assets and a Civitai LoRA into ComfyUI.

import os
import subprocess
import urllib.parse
import requests
import getpass
from email.message import Message
import gdown

WORKSPACE = "/content/ComfyUI"

# =========================================================
# INPUT RESOURCES
# =========================================================

UNET_URLS = "https://huggingface.co/unsloth/Z-Image-Turbo-GGUF/resolve/main/z-image-turbo-Q8_0.gguf" # @param {type:"string"}

TEXT_ENCODER_URLS = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors" # @param {type:"string"}

VAE_URLS = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors" # @param {type:"string"}

# Civitai LoRA details
CIVITAI_MODEL_ID = "2687183" # @param {type:"string"}
CIVITAI_FILE_ID = "2572837" # @param {type:"string"}

# Optional non-Civitai LoRA URLs.
# Leave blank if using the Civitai downloader above.
LORA_URLS = "https://civitai.com/api/download/models/2687183?token=a53f74995c7df841e2434a648dee2cdf&fileId=2572837" # @param {type:"string"}

# =========================================================
# DESTINATION FOLDERS
# =========================================================

DIRS = {
    "unet": os.path.join(WORKSPACE, "models/unet"),
    "clip": os.path.join(WORKSPACE, "models/clip"),
    "vae": os.path.join(WORKSPACE, "models/vae"),
    "loras": os.path.join(WORKSPACE, "models/loras"),
}

for folder in DIRS.values():
    os.makedirs(folder, exist_ok=True)

print("⚡ Configuring downloader...")

# Install aria2 if needed
subprocess.run(
    ["apt-get", "-y", "install", "-qq", "aria2"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# =========================================================
# FILENAME HELPER
# =========================================================

def get_filename(response, fallback):
    cd = response.headers.get("Content-Disposition", "")
    msg = Message()
    msg["Content-Disposition"] = cd
    filename = msg.get_filename()

    if filename:
        return os.path.basename(filename)

    # Try redirected URL filename
    name = os.path.basename(
        urllib.parse.urlparse(response.url).path
    )

    if name and "." in name:
        return name

    return fallback

# =========================================================
# AUTHENTICATED CIVITAI DOWNLOADER
# =========================================================

def download_civitai(model_id, file_id, target_dir, token):

    url = f"https://civitai.com/api/download/models/{model_id}"
    params = {"token": token, "fileId": file_id}

    print("\n📥 Connecting to Civitai...")

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Authorization": f"Bearer {token}"
    }

    with requests.get(
        url,
        params=params,
        headers=headers,
        stream=True,
        allow_redirects=True,
        timeout=(30, 180)
    ) as response:

        print("   HTTP Status:", response.status_code)
        response.raise_for_status()

        filename = get_filename(
            response,
            f"civitai_{model_id}_{file_id}.safetensors"
        )

        filepath = os.path.join(target_dir, filename)
        total = 0

        with open(filepath, "wb") as f:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)
                    total += len(chunk)

        if total == 0:
            if os.path.exists(filepath):
                os.remove(filepath)
            raise Exception("Downloaded file is empty")

        print(
            f"   ✅ Saved: {filename} "
            f"({total / 1024 / 1024:.1f} MB)"
        )

# =========================================================
# GENERIC DOWNLOADER
# =========================================================

def download_file(url, target_dir):

    try:
        os.makedirs(target_dir, exist_ok=True)

        if "drive.google.com" in url:
            print("   📥 Downloading from Google Drive...")
            gdown.download(
                url,
                output=target_dir + "/",
                quiet=False,
                fuzzy=True
            )

        elif "civitai.com" in urllib.parse.urlparse(url).netloc:
            print(
                "   ⚠️ Civitai links are handled by "
                "the authenticated downloader."
            )

        else:
            parsed_url = urllib.parse.urlparse(url)
            filename = os.path.basename(parsed_url.path)

            print(f"   📥 Fetching: {filename[:60]}...")

            cmd = [
                "aria2c",
                "--console-log-level=notice",
                "--summary-interval=10",
                "-c",
                "-x", "4",
                "-s", "4",
                "-k", "1M",
                "--retry-wait=5",
                "--max-tries=5",
                url,
                "-d", target_dir
            ]

            if filename:
                cmd.extend(["-o", filename])

            subprocess.run(cmd, check=True)
            print("   ✅ Download complete")

    except Exception as e:
        print("   ❌ Download failed:", str(e))

# =========================================================
# PROCESS URL LISTS
# =========================================================

def process_downloads(urls_str, target_dir):

    if not urls_str.strip():
        return

    url_list = [
        u.strip()
        for u in urls_str.replace(",", "\n").split("\n")
        if u.strip()
    ]

    print(f"\n📂 Directory: {os.path.basename(target_dir)}")

    for url in url_list:
        download_file(url, target_dir)

# =========================================================
# START DOWNLOADS
# =========================================================

# 1. UNET
process_downloads(UNET_URLS, DIRS["unet"])

# 2. CIVITAI LORA - AUTHENTICATED DOWNLOAD
if CIVITAI_MODEL_ID.strip() and CIVITAI_FILE_ID.strip():

    print("\n📂 Directory: loras")

    api_token = getpass.getpass(
        "Enter your NEW Civitai API key (hidden): "
    )

    if api_token.strip():
        try:
            download_civitai(
                CIVITAI_MODEL_ID.strip(),
                CIVITAI_FILE_ID.strip(),
                DIRS["loras"],
                api_token.strip()
            )
        except Exception as e:
            print("   ❌ Civitai download failed:", str(e))
    else:
        print("   ⚠️ No API key entered. Skipping Civitai.")

# 3. OPTIONAL OTHER LORA URLS
process_downloads(LORA_URLS, DIRS["loras"])

# 4. TEXT ENCODER
process_downloads(TEXT_ENCODER_URLS, DIRS["clip"])

# 5. VAE
process_downloads(VAE_URLS, DIRS["vae"])

print("\n✅ Asset download process finished!")

print("\n📁 Downloaded model folders:")
for name, folder in DIRS.items():
    print(f"\n{name.upper()}:")
    for filename in os.listdir(folder):
        print("  -", filename)


⚡ Configuring downloader...

📂 Directory: unet
   📥 Fetching: z-image-turbo-Q8_0.gguf...
   ✅ Download complete

📂 Directory: loras
Enter your NEW Civitai API key (hidden): ··········

📥 Connecting to Civitai...
   HTTP Status: 200
   ✅ Saved: gym_fit_zit.safetensors (48.7 MB)

📂 Directory: loras
   ⚠️ Civitai links are handled by the authenticated downloader.

📂 Directory: clip
   📥 Fetching: qwen_3_4b.safetensors...
   ✅ Download complete

📂 Directory: vae
   📥 Fetching: ae.safetensors...
   ✅ Download complete

✅ Asset download process finished!

📁 Downloaded model folders:

UNET:
  - put_unet_files_here
  - z-image-turbo-Q8_0.gguf

CLIP:
  - qwen_3_4b.safetensors
  - put_clip_or_text_encoder_models_here

VAE:
  - ae.safetensors
  - put_vae_here

LORAS:
  - put_loras_here
  - gym_fit_zit.safetensors


In [ ]:

#@title 3. Image Generation — Model Dropdown

import os
import sys
import json
import time
import random
import subprocess
import urllib.request
import urllib.error
import ipywidgets as widgets
from IPython.display import display, Image as IPImage

WORKSPACE = "/content/ComfyUI"

# Find installed models
unet_folder = os.path.join(WORKSPACE, "models", "unet")
lora_folder = os.path.join(WORKSPACE, "models", "loras")

def get_files(folder, extensions):
    if not os.path.isdir(folder):
        return []
    return sorted(
        f for f in os.listdir(folder)
        if f.lower().endswith(extensions)
    )

unet_models = get_files(unet_folder, (".gguf", ".safetensors"))
lora_models = get_files(lora_folder, (".safetensors",))

if not unet_models:
    raise FileNotFoundError(
        f"No UNET models found in {unet_folder}. Install a model first."
    )

# Dropdowns
unet_dropdown = widgets.Dropdown(
    options=unet_models,
    description="UNET:",
    layout=widgets.Layout(width="600px")
)

lora_dropdown = widgets.Dropdown(
    options=["None"] + lora_models,
    description="LoRA:",
    layout=widgets.Layout(width="600px")
)

strength_slider = widgets.FloatSlider(
    value=1.0, min=0, max=2, step=0.05,
    description="LoRA strength:",
    readout_format=".2f",
    layout=widgets.Layout(width="600px")
)

prompt_box = widgets.Text(
    value="gym fit, hot milf having fun",
    description="Prompt:",
    layout=widgets.Layout(width="600px")
)

negative_box = widgets.Text(
    value="blurry, low quality, deformed, artifacts",
    description="Negative:",
    layout=widgets.Layout(width="600px")
)

display(widgets.VBox([
    unet_dropdown,
    lora_dropdown,
    strength_slider,
    prompt_box,
    negative_box
]))

# Generation settings
WIDTH = 1024
HEIGHT = 1024
BATCH_SIZE = 1
STEPS = 9
CFG = 1.0
SAMPLER_NAME = "res_multistep"
SCHEDULER = "beta"
AURA_SHIFT = 3.0
SEED = random.randint(1, 1125899906842624)

# Start ComfyUI
os.chdir(WORKSPACE)
print("Checking ComfyUI server...")

def server_ready():
    try:
        urllib.request.urlopen(
            "http://127.0.0.1:8188", timeout=3
        )
        return True
    except Exception:
        return False

if not server_ready():
    print("Starting ComfyUI...")
    subprocess.Popen(
        [sys.executable, "main.py"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    for _ in range(90):
        if server_ready():
            break
        time.sleep(2)

if not server_ready():
    raise RuntimeError("ComfyUI did not start. Check its installation.")

print("ComfyUI is ready.")

# Click this button after selecting models and entering prompt
generate_button = widgets.Button(
    description="Generate Image",
    button_style="success",
    icon="image"
)

output_area = widgets.Output()

def generate_image(_):
    with output_area:
        output_area.clear_output()

        UNET_FILENAME = unet_dropdown.value
        LORA_FILENAME = lora_dropdown.value
        LORA_STRENGTH = strength_slider.value
        PROMPT = prompt_box.value
        NEGATIVE_PROMPT = negative_box.value

        if not PROMPT.strip():
            print("Please enter a prompt.")
            return

        print("Selected UNET:", UNET_FILENAME)
        print("Selected LoRA:", LORA_FILENAME)
        print("Submitting generation...")

        workflow = {
            "9": {
                "inputs": {
                    "filename_prefix": "z-image",
                    "images": ["43", 0]
                },
                "class_type": "SaveImage"
            },
            "39": {
                "inputs": {
                    "clip_name": "qwen_3_4b.safetensors",
                    "type": "lumina2",
                    "device": "default"
                },
                "class_type": "CLIPLoader"
            },
            "40": {
                "inputs": {"vae_name": "ae.safetensors"},
                "class_type": "VAELoader"
            },
            "41": {
                "inputs": {
                    "width": WIDTH,
                    "height": HEIGHT,
                    "batch_size": BATCH_SIZE
                },
                "class_type": "EmptySD3LatentImage"
            },
            "42": {
                "inputs": {
                    "text": NEGATIVE_PROMPT,
                    "clip": ["39", 0]
                },
                "class_type": "CLIPTextEncode"
            },
            "43": {
                "inputs": {
                    "samples": ["44", 0],
                    "vae": ["40", 0]
                },
                "class_type": "VAEDecode"
            },
            "44": {
                "inputs": {
                    "seed": SEED,
                    "steps": STEPS,
                    "cfg": CFG,
                    "sampler_name": SAMPLER_NAME,
                    "scheduler": SCHEDULER,
                    "denoise": 1,
                    "model": ["47", 0],
                    "positive": ["45", 0],
                    "negative": ["42", 0],
                    "latent_image": ["41", 0]
                },
                "class_type": "KSampler"
            },
            "45": {
                "inputs": {
                    "text": PROMPT,
                    "clip": ["39", 0]
                },
                "class_type": "CLIPTextEncode"
            }
        }

        # Load selected UNET
        if UNET_FILENAME.lower().endswith(".gguf"):
            workflow["48"] = {
                "inputs": {"unet_name": UNET_FILENAME},
                "class_type": "UnetLoaderGGUF"
            }
        else:
            workflow["48"] = {
                "inputs": {
                    "unet_name": UNET_FILENAME,
                    "weight_dtype": "default"
                },
                "class_type": "UNETLoader"
            }

        # Optional LoRA
        if LORA_FILENAME != "None":
            workflow["50"] = {
                "inputs": {
                    "lora_name": LORA_FILENAME,
                    "strength_model": LORA_STRENGTH,
                    "model": ["48", 0]
                },
                "class_type": "LoraLoaderModelOnly"
            }
            model_node = ["50", 0]
        else:
            model_node = ["48", 0]

        workflow["47"] = {
            "inputs": {
                "shift": AURA_SHIFT,
                "model": model_node
            },
            "class_type": "ModelSamplingAuraFlow"
        }

        payload = json.dumps({"prompt": workflow}).encode("utf-8")
        request = urllib.request.Request(
            "http://127.0.0.1:8188/prompt",
            data=payload,
            headers={"Content-Type": "application/json"}
        )

        try:
            response = urllib.request.urlopen(request)
            prompt_id = json.loads(response.read())["prompt_id"]
        except urllib.error.HTTPError as e:
            print("ComfyUI API error:", e.code)
            print(e.read().decode("utf-8", errors="replace"))
            return

        print("Generation started. Please wait...")

        for _ in range(600):
            try:
                req = urllib.request.Request(
                    f"http://127.0.0.1:8188/history/{prompt_id}"
                )
                history = json.loads(
                    urllib.request.urlopen(req).read()
                )

                if prompt_id in history:
                    entry = history[prompt_id]

                    if entry.get("status", {}).get("status_str") == "error":
                        print("Generation failed:")
                        print(entry.get("status", {}).get("messages", []))
                        return

                    outputs = entry.get("outputs", {})
                    break
            except Exception:
                pass

            time.sleep(2)
        else:
            print("Timed out waiting for generation.")
            return

        for node_output in outputs.values():
            for img in node_output.get("images", []):
                filename = img["filename"]
                subfolder = img.get("subfolder", "")
                img_path = os.path.join(
                    WORKSPACE, "output", subfolder, filename
                )

                print("Saved:", img_path)
                display(IPImage(filename=img_path))

generate_button.on_click(generate_image)
display(generate_button, output_area)


Checking ComfyUI server...
Starting ComfyUI...
ComfyUI is ready.


Button(button_style='success', description='Generate Image', icon='image', style=ButtonStyle())

Output()

In [ ]:
 #@title 4. Export & Download Results
#@markdown Run this to instantly zip and download all the generated images from this session.

import os
from google.colab import files

OUTPUT_DIR = "/content/ComfyUI/output"
ZIP_NAME = "/content/Z_Image_Artworks.zip"

if os.path.exists(OUTPUT_DIR) and len(os.listdir(OUTPUT_DIR)) > 0:
    print("🗜️ Zipping generated artworks...")
    !zip -j -q {ZIP_NAME} {OUTPUT_DIR}/*.png
    print("📥 Initiating download...")
    files.download(ZIP_NAME)
else:
    print("⚠️ No images found in the output directory yet!")

In [ ]:

# @title 3. Install GGUF Support and Start ComfyUI

import os
import subprocess
import time
import requests

WORKSPACE = "/content/ComfyUI"

# Check that the downloaded models exist
required_files = {
    "UNET": f"{WORKSPACE}/models/unet/z-image-turbo-Q8_0.gguf",
    "TEXT ENCODER": f"{WORKSPACE}/models/clip/qwen_3_4b.safetensors",
    "VAE": f"{WORKSPACE}/models/vae/ae.safetensors",
    "LORA": f"{WORKSPACE}/models/loras/gym_fit_zit.safetensors",
}

for name, path in required_files.items():
    if os.path.isfile(path):
        print(f"✅ {name}: {os.path.basename(path)}")
    else:
        raise FileNotFoundError(f"Missing {name}: {path}")

# Install ComfyUI-GGUF custom node
custom_nodes = f"{WORKSPACE}/custom_nodes"
gguf_path = f"{custom_nodes}/ComfyUI-GGUF"

os.makedirs(custom_nodes, exist_ok=True)

if not os.path.isdir(gguf_path):
    subprocess.run([
        "git", "clone",
        "https://github.com/city96/ComfyUI-GGUF.git",
        gguf_path
    ], check=True)
else:
    print("GGUF custom node already exists.")

# Install custom-node requirements
requirements = f"{gguf_path}/requirements.txt"
if os.path.isfile(requirements):
    subprocess.run([
        "pip", "install", "-r", requirements
    ], check=True)

# Start ComfyUI if it is not already running
try:
    r = requests.get("http://127.0.0.1:8188/system_stats", timeout=3)
    running = r.status_code == 200
except Exception:
    running = False

if not running:
    log_file = open("/content/comfyui.log", "w")
    process = subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=WORKSPACE,
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

    for i in range(60):
        time.sleep(2)
        try:
            r = requests.get(
                "http://127.0.0.1:8188/system_stats",
                timeout=3
            )
            if r.status_code == 200:
                running = True
                break
        except Exception:
            pass

if running:
    print("✅ ComfyUI is running at http://127.0.0.1:8188")
else:
    print("❌ ComfyUI did not start. Check /content/comfyui.log")


✅ UNET: z-image-turbo-Q8_0.gguf
✅ TEXT ENCODER: qwen_3_4b.safetensors
✅ VAE: ae.safetensors
✅ LORA: gym_fit_zit.safetensors
GGUF custom node already exists.
✅ ComfyUI is running at http://127.0.0.1:8188


In [ ]:

# @title 2B. Download Two Additional Civitai LoRAs

import os
import requests
import getpass
from email.message import Message

LORA_DIR = "/content/ComfyUI/models/loras"
os.makedirs(LORA_DIR, exist_ok=True)

NEW_LORAS = [
    {
        "model_id": "2606654",
        "file_id": "2493976"
    },
    {
        "model_id": "2606598",
        "file_id": "2493919"
    }
]

api_key = getpass.getpass(
    "Enter your NEW Civitai API key (hidden): "
).strip()

if not api_key:
    raise ValueError("API key is required.")

headers = {
    "Authorization": f"Bearer {api_key}",
    "User-Agent": "Mozilla/5.0"
}

for item in NEW_LORAS:

    model_id = item["model_id"]
    file_id = item["file_id"]

    url = f"https://civitai.com/api/download/models/{model_id}"

    params = {
        "fileId": file_id,
        "token": api_key
    }

    print(f"\n📥 Downloading model {model_id}, file {file_id}...")

    try:
        with requests.get(
            url,
            params=params,
            headers=headers,
            stream=True,
            allow_redirects=True,
            timeout=(30, 180)
        ) as response:

            response.raise_for_status()

            msg = Message()
            msg["Content-Disposition"] = response.headers.get(
                "Content-Disposition", ""
            )

            filename = msg.get_filename()

            if not filename:
                filename = f"civitai_{model_id}_{file_id}.safetensors"

            filename = os.path.basename(filename)
            filepath = os.path.join(LORA_DIR, filename)

            if os.path.exists(filepath):
                print(f"⏭️ Already exists: {filename}")
                continue

            total = 0

            with open(filepath, "wb") as f:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        total += len(chunk)

            if total == 0:
                if os.path.exists(filepath):
                    os.remove(filepath)
                raise RuntimeError("Empty download.")

            print(
                f"✅ Downloaded {filename} "
                f"({total / 1024 / 1024:.1f} MB)"
            )

    except Exception as e:
        print(f"❌ Download failed: {e}")

print("\n========== ALL AVAILABLE LORAS ==========")

for filename in sorted(os.listdir(LORA_DIR)):
    if filename.lower().endswith(".safetensors"):
        print("✅", filename)

print("\nDownload check finished.")


Enter your NEW Civitai API key (hidden): ··········

📥 Downloading model 2606654, file 2493976...
✅ Downloaded Ridhika Aunty.safetensors (162.2 MB)

📥 Downloading model 2606598, file 2493919...
✅ Downloaded Amira.safetensors (162.2 MB)

========== ALL AVAILABLE LORAS ==========
✅ 20261024-89544939.safetensors
✅ Amira.safetensors
✅ Ridhika Aunty.safetensors
✅ TurboPussyZ_v2.safetensors
✅ Western_Woman_Boobs_1_Z-Image Turbo_000004500.safetensors
✅ gym_fit_zit.safetensors

Download check finished.


In [ ]:

# @title Refresh All LoRA Dropdown Options

LORA_DIR = "/content/ComfyUI/models/loras"

lora_files = sorted([
    f for f in os.listdir(LORA_DIR)
    if f.lower().endswith(".safetensors")
])

lora_dropdown.options = ["None"] + lora_files
lora_dropdown.value = "None"

print(f"✅ Found {len(lora_files)} LoRA files:")
for filename in lora_files:
    print("•", filename)



✅ Found 6 LoRA files:
• 20261024-89544939.safetensors
• Amira.safetensors
• Ridhika Aunty.safetensors
• TurboPussyZ_v2.safetensors
• Western_Woman_Boobs_1_Z-Image Turbo_000004500.safetensors
• gym_fit_zit.safetensors


In [ ]:

# @title Download All 4 Civitai LoRAs

import os
import requests
import getpass
from email.message import Message

LORA_DIR = "/content/ComfyUI/models/loras"
os.makedirs(LORA_DIR, exist_ok=True)

# All four Civitai models
LORAS = [
    {
        "name": "gym_fit_zit.safetensors",
        "model_id": "2687183",
        "file_id": "2572837"
    },
    {
        "name": "20261024_89544939.safetensors",
        "model_id": "3425492",
        "file_id": "3320359"
    },
    {
        "name": "Western_Woman_Boobs_1_Z-Image Turbo_000004500.safetensors",
        "model_id": "2639284",
        "file_id": "2527219"
    },
    {
        "name": "Civitai_3298134_3183010.safetensors",
        "model_id": "3298134",
        "file_id": "3183010"
    }
]

# Enter your NEW Civitai API key
api_key = getpass.getpass(
    "Enter your new Civitai API key (hidden): "
).strip()

if not api_key:
    raise ValueError("Civitai API key is required.")

headers = {
    "Authorization": f"Bearer {api_key}",
    "User-Agent": "Mozilla/5.0"
}

def download_lora(model):

    model_id = model["model_id"]
    file_id = model["file_id"]
    expected_name = model["name"]

    print(f"\n📥 Processing: {expected_name}")

    url = f"https://civitai.com/api/download/models/{model_id}"

    params = {
        "fileId": file_id,
        "token": api_key
    }

    with requests.get(
        url,
        params=params,
        headers=headers,
        stream=True,
        allow_redirects=True,
        timeout=(30, 180)
    ) as response:

        response.raise_for_status()

        # Get original filename from Civitai
        msg = Message()
        msg["Content-Disposition"] = response.headers.get(
            "Content-Disposition", ""
        )

        filename = msg.get_filename()

        if not filename:
            filename = expected_name

        filename = os.path.basename(filename)

        if not filename.lower().endswith(".safetensors"):
            filename = expected_name

        filepath = os.path.join(LORA_DIR, filename)

        # Skip an already downloaded file
        if os.path.isfile(filepath):
            print(f"⏭️ Already exists: {filename}")
            return

        total = 0

        with open(filepath, "wb") as f:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)
                    total += len(chunk)

        if total == 0:
            if os.path.exists(filepath):
                os.remove(filepath)
            raise RuntimeError("Downloaded file is empty.")

        print(
            f"✅ Downloaded: {filename} "
            f"({total / 1024 / 1024:.1f} MB)"
        )


# Download all four
for model in LORAS:
    try:
        download_lora(model)
    except Exception as error:
        print(f"❌ Failed: {model['name']}")
        print("Reason:", error)


# Show installed LoRAs
print("\n========== ALL INSTALLED LORAS ==========")

for filename in sorted(os.listdir(LORA_DIR)):
    if filename.lower().endswith(".safetensors"):
        size = os.path.getsize(
            os.path.join(LORA_DIR, filename)
        ) / (1024 * 1024)

        print(f"✅ {filename} ({size:.1f} MB)")

print("\n✅ LoRA download check finished!")


Enter your new Civitai API key (hidden): ··········

📥 Processing: gym_fit_zit.safetensors
⏭️ Already exists: gym_fit_zit.safetensors

📥 Processing: 20261024_89544939.safetensors
✅ Downloaded: 20261024-89544939.safetensors (81.2 MB)

📥 Processing: Western_Woman_Boobs_1_Z-Image Turbo_000004500.safetensors
✅ Downloaded: TurboPussyZ_v2.safetensors (162.2 MB)

📥 Processing: Civitai_3298134_3183010.safetensors
✅ Downloaded: Western_Woman_Boobs_1_Z-Image Turbo_000004500.safetensors (162.2 MB)

========== ALL INSTALLED LORAS ==========
✅ 20261024-89544939.safetensors (81.2 MB)
✅ TurboPussyZ_v2.safetensors (162.2 MB)
✅ Western_Woman_Boobs_1_Z-Image Turbo_000004500.safetensors (162.2 MB)
✅ gym_fit_zit.safetensors (48.7 MB)

✅ LoRA download check finished!


In [ ]:

# @title 🎨 Z-IMAGE TURBO STUDIO — Complete Image Generation Panel

import os
import time
import uuid
import requests
import getpass
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output, Image

# =========================================================
# CONFIGURATION
# =========================================================

WORKSPACE = "/content/ComfyUI"
API = "http://127.0.0.1:8188"
LORA_DIR = f"{WORKSPACE}/models/loras"

os.makedirs(LORA_DIR, exist_ok=True)

# Four known Civitai model versions
LORA_INFO = {
    "gym_fit_zit.safetensors": {
        "version_id": "2572837",
        "model_id": "2687183"
    },
    "20261024_89544939.safetensors": {
        "version_id": "3320359",
        "model_id": "3425492"
    },
    "Western_Woman_Boobs_1_Z-Image Turbo_000004500.safetensors": {
        "version_id": "2527219",
        "model_id": "2639284"
    },
    "Civitai_3298134_3183010.safetensors": {
        "version_id": "3183010",
        "model_id": "3298134"
    }
}

# =========================================================
# CHECK COMFYUI
# =========================================================

try:
    server_check = requests.get(
        f"{API}/system_stats", timeout=5
    )
    server_check.raise_for_status()
except Exception:
    raise RuntimeError(
        "ComfyUI is not running. Run Cell 3 first."
    )

# Check required model files
required_files = [
    f"{WORKSPACE}/models/unet/z-image-turbo-Q8_0.gguf",
    f"{WORKSPACE}/models/clip/qwen_3_4b.safetensors",
    f"{WORKSPACE}/models/vae/ae.safetensors"
]

for path in required_files:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Missing model: {path}")

# =========================================================
# CHECK AVAILABLE COMFYUI NODES
# =========================================================

object_response = requests.get(
    f"{API}/object_info", timeout=60
)
object_response.raise_for_status()
NODE_INFO = object_response.json()

required_nodes = [
    "UnetLoaderGGUF",
    "CLIPLoader",
    "VAELoader",
    "CLIPTextEncode",
    "EmptySD3LatentImage",
    "ModelSamplingAuraFlow",
    "KSampler",
    "VAEDecode",
    "SaveImage"
]

missing_nodes = [
    node for node in required_nodes
    if node not in NODE_INFO
]

if missing_nodes:
    raise RuntimeError(
        "Missing ComfyUI nodes: " + ", ".join(missing_nodes)
    )

if "LoraLoaderModelOnly" not in NODE_INFO:
    raise RuntimeError(
        "LoraLoaderModelOnly node is missing."
    )

# Read available sampler and scheduler choices
sampler_options = NODE_INFO["KSampler"]["input"]["required"]["sampler_name"][0]
scheduler_options = NODE_INFO["KSampler"]["input"]["required"]["scheduler"][0]

sampler_default = (
    "res_multistep"
    if "res_multistep" in sampler_options
    else "euler"
)

scheduler_default = (
    "beta"
    if "beta" in scheduler_options
    else "simple"
)

# =========================================================
# CIVITAI KEYWORDS
# =========================================================

print("Enter your Civitai API key to retrieve published trigger words.")
civitai_key = getpass.getpass(
    "Civitai API key (hidden, optional): "
).strip()

keyword_cache = {}

def fetch_keywords(filename):

    if filename in keyword_cache:
        return keyword_cache[filename]

    info = LORA_INFO.get(filename)

    if not info or not civitai_key:
        keyword_cache[filename] = []
        return []

    try:
        response = requests.get(
            f"https://civitai.com/api/v1/model-versions/{info['version_id']}",
            headers={
                "Authorization": f"Bearer {civitai_key}"
            },
            timeout=25
        )

        response.raise_for_status()

        data = response.json()
        words = data.get("trainedWords") or []

        keyword_cache[filename] = words
        return words

    except Exception as e:
        print(f"Keyword lookup failed for {filename}: {e}")
        keyword_cache[filename] = []
        return []

# =========================================================
# REFRESH LORA LIST
# =========================================================

def get_lora_files():

    return sorted([
        f for f in os.listdir(LORA_DIR)
        if f.lower().endswith(".safetensors")
    ])

lora_files = get_lora_files()

if not lora_files:
    raise RuntimeError("No LoRA files found in your LoRA folder.")

# =========================================================
# BEAUTIFUL INTERFACE
# =========================================================

display(HTML("""
<style>
.studio-title {
    background: linear-gradient(120deg,#17152d,#30215a,#543a8c);
    color: white;
    padding: 24px;
    border-radius: 16px;
    margin-bottom: 14px;
}
.studio-title h1 {
    color: white;
    font-size: 27px;
    margin: 0 0 7px 0;
}
.studio-title p {
    color: #ded5ff;
    margin: 0;
}
.studio-section {
    background: #20212a;
    border: 1px solid #414252;
    border-radius: 12px;
    padding: 13px;
    margin: 8px 0;
    color: #eeeeff;
}
.studio-section h3 {
    color: #c9b5ff;
    margin-top: 0;
}
.studio-note {
    color: #a9a9bb;
    font-size: 12px;
}
</style>

<div class="studio-title">
  <h1>✦ Z-IMAGE TURBO STUDIO</h1>
  <p>AI Image Creation • LoRA Control • Advanced Generation Settings</p>
</div>
"""))

# Common widget layout
wide = widgets.Layout(width="100%")

def section_title(title):
    return widgets.HTML(
        f'<div class="studio-section"><h3>{title}</h3></div>'
    )

# =========================================================
# MODEL SELECTION
# =========================================================

unet_dropdown = widgets.Dropdown(
    options=[
        f for f in os.listdir(f"{WORKSPACE}/models/unet")
        if f.lower().endswith(".gguf")
    ],
    value="z-image-turbo-Q8_0.gguf",
    description="UNET:",
    layout=wide,
    style={"description_width": "90px"}
)

lora_dropdown = widgets.Dropdown(
    options=["None"] + lora_files,
    value="None",
    description="LoRA:",
    layout=wide,
    style={"description_width": "90px"}
)

lora_strength = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=2.0,
    step=0.05,
    description="Strength:",
    readout_format=".2f",
    layout=wide,
    style={"description_width": "90px"}
)

auto_keywords = widgets.Checkbox(
    value=True,
    description="Automatically add selected LoRA keywords to prompt",
    indent=False,
    layout=wide
)

keyword_display = widgets.Textarea(
    value="Select a LoRA to view its published trigger words.",
    description="Keywords:",
    disabled=True,
    layout=widgets.Layout(width="100%", height="85px"),
    style={"description_width": "90px"}
)

keyword_status = widgets.HTML(
    value='<span style="color:#aaa">Select a LoRA.</span>'
)

# =========================================================
# PROMPT SETTINGS
# =========================================================

prompt_box = widgets.Textarea(
    value="photorealistic portrait, beautiful adult woman, athletic body, detailed face, realistic skin, professional photography, natural lighting",
    placeholder="Describe your image...",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="145px"),
    style={"description_width": "90px"}
)

negative_box = widgets.Textarea(
    value="blurry, low quality, distorted face, extra fingers, bad anatomy, deformed hands, watermark, text",
    description="Negative:",
    layout=widgets.Layout(width="100%", height="95px"),
    style={"description_width": "90px"}
)

# =========================================================
# IMAGE SETTINGS
# =========================================================

width_widget = widgets.Dropdown(
    options=[512, 640, 768, 896, 1024, 1152, 1280],
    value=768,
    description="Width:",
    layout=widgets.Layout(width="48%")
)

height_widget = widgets.Dropdown(
    options=[512, 640, 768, 896, 1024, 1152, 1280],
    value=1024,
    description="Height:",
    layout=widgets.Layout(width="48%")
)

steps_widget = widgets.IntSlider(
    value=9,
    min=4,
    max=30,
    step=1,
    description="Steps:",
    layout=wide
)

cfg_widget = widgets.FloatSlider(
    value=1.0,
    min=1.0,
    max=7.0,
    step=0.5,
    description="CFG:",
    readout_format=".1f",
    layout=wide
)

seed_widget = widgets.IntText(
    value=0,
    description="Seed:",
    layout=wide
)

sampler_widget = widgets.Dropdown(
    options=sampler_options,
    value=sampler_default,
    description="Sampler:",
    layout=wide
)

scheduler_widget = widgets.Dropdown(
    options=scheduler_options,
    value=scheduler_default,
    description="Scheduler:",
    layout=wide
)

# =========================================================
# KEYWORD PANEL
# =========================================================

def update_lora_keywords(change=None):

    selected = lora_dropdown.value

    if selected == "None":
        keyword_display.value = "No LoRA selected."
        keyword_status.value = ""
        return

    words = fetch_keywords(selected)

    if words:
        keyword_display.value = ", ".join(words)
        keyword_status.value = (
            '<span style="color:#75e6a0">✓ Published keywords found.</span>'
        )

        if auto_keywords.value:
            current = prompt_box.value.strip()
            existing = current.lower()

            missing = [
                word for word in words
                if word.lower() not in existing
            ]

            if missing:
                prompt_box.value = (
                    current + ", " + ", ".join(missing)
                ).strip(", ")

    else:
        keyword_display.value = (
            "No published trigger words were returned by Civitai. "
            "You can enter the trigger words manually."
        )
        keyword_status.value = (
            '<span style="color:#e5b66d">No metadata available.</span>'
        )

lora_dropdown.observe(update_lora_keywords, names="value")

# =========================================================
# GENERATION
# =========================================================

generate_button = widgets.Button(
    description="✦ GENERATE IMAGE",
    button_style="success",
    icon="image",
    layout=widgets.Layout(width="100%", height="48px")
)

refresh_button = widgets.Button(
    description="↻ Refresh LoRA List",
    button_style="info",
    layout=widgets.Layout(width="48%")
)

status = widgets.HTML(
    value='<span style="color:#aaa">Ready to generate.</span>'
)

output = widgets.Output()

def refresh_loras(_):

    global lora_files

    lora_files = get_lora_files()
    lora_dropdown.options = ["None"] + lora_files

    status.value = (
        '<span style="color:#75e6a0">LoRA list refreshed.</span>'
    )

refresh_button.on_click(refresh_loras)

def generate_image(_):

    generate_button.disabled = True
    status.value = (
        '<span style="color:#e5c477">⏳ Preparing generation...</span>'
    )

    with output:
        clear_output()

    try:
        selected_lora = lora_dropdown.value

        # Add selected LoRA keywords if enabled
        final_prompt = prompt_box.value.strip()

        if selected_lora != "None" and auto_keywords.value:
            words = fetch_keywords(selected_lora)

            for word in words:
                if word.lower() not in final_prompt.lower():
                    final_prompt += ", " + word

        # Construct workflow
        workflow = {
            "1": {
                "class_type": "UnetLoaderGGUF",
                "inputs": {
                    "unet_name": unet_dropdown.value
                }
            },

            "2": {
                "class_type": "CLIPLoader",
                "inputs": {
                    "clip_name": "qwen_3_4b.safetensors",
                    "type": "lumina2",
                    "device": "default"
                }
            },

            "3": {
                "class_type": "VAELoader",
                "inputs": {
                    "vae_name": "ae.safetensors"
                }
            },

            "4": {
                "class_type": "CLIPTextEncode",
                "inputs": {
                    "text": final_prompt,
                    "clip": ["2", 0]
                }
            },

            "5": {
                "class_type": "CLIPTextEncode",
                "inputs": {
                    "text": negative_box.value,
                    "clip": ["2", 0]
                }
            },

            "6": {
                "class_type": "EmptySD3LatentImage",
                "inputs": {
                    "width": width_widget.value,
                    "height": height_widget.value,
                    "batch_size": 1
                }
            },

            "7": {
                "class_type": "ModelSamplingAuraFlow",
                "inputs": {
                    "model": ["1", 0],
                    "shift": 3.0
                }
            },

            "8": {
                "class_type": "KSampler",
                "inputs": {
                    "seed": (
                        seed_widget.value
                        if seed_widget.value > 0
                        else int(time.time() * 1000) % (2**32)
                    ),
                    "steps": steps_widget.value,
                    "cfg": cfg_widget.value,
                    "sampler_name": sampler_widget.value,
                    "scheduler": scheduler_widget.value,
                    "denoise": 1.0,
                    "model": ["7", 0],
                    "positive": ["4", 0],
                    "negative": ["5", 0],
                    "latent_image": ["6", 0]
                }
            },

            "9": {
                "class_type": "VAEDecode",
                "inputs": {
                    "samples": ["8", 0],
                    "vae": ["3", 0]
                }
            },

            "10": {
                "class_type": "SaveImage",
                "inputs": {
                    "filename_prefix": "ZImageStudio",
                    "images": ["9", 0]
                }
            }
        }

        # Add LoRA when selected
        if selected_lora != "None":

            workflow["11"] = {
                "class_type": "LoraLoaderModelOnly",
                "inputs": {
                    "model": ["7", 0],
                    "lora_name": selected_lora,
                    "strength_model": lora_strength.value
                }
            }

            workflow["8"]["inputs"]["model"] = ["11", 0]

        # Submit to ComfyUI
        payload = {
            "prompt": workflow,
            "client_id": str(uuid.uuid4())
        }

        response = requests.post(
            f"{API}/prompt",
            json=payload,
            timeout=60
        )

        if not response.ok:
            raise RuntimeError(response.text[:3000])

        prompt_id = response.json()["prompt_id"]

        status.value = (
            '<span style="color:#e5c477">⏳ Generating image...</span>'
        )

        # Wait for completed output
        image_info = None

        for attempt in range(600):

            time.sleep(2)

            history_response = requests.get(
                f"{API}/history/{prompt_id}",
                timeout=20
            )

            history_response.raise_for_status()
            history = history_response.json()

            if prompt_id in history:

                entry = history[prompt_id]
                outputs = entry.get("outputs", {})

                if "10" in outputs:
                    images = outputs["10"].get("images", [])

                    if images:
                        image_info = images[0]
                        break

                if entry.get("status", {}).get("status_str") == "error":
                    raise RuntimeError(
                        str(entry.get("status", {}))
                    )

        if not image_info:
            raise TimeoutError(
                "No image returned after waiting. "
                "Check the ComfyUI log."
            )

        # Retrieve generated image
        image_response = requests.get(
            f"{API}/view",
            params={
                "filename": image_info["filename"],
                "subfolder": image_info.get("subfolder", ""),
                "type": image_info.get("type", "output")
            },
            timeout=60
        )

        image_response.raise_for_status()

        status.value = (
            '<span style="color:#75e6a0">✓ Image generated successfully!</span>'
        )

        with output:
            clear_output()
            display(HTML("<h3>✨ Your Generated Image</h3>"))
            display(Image(data=image_response.content))

    except Exception as e:

        status.value = (
            '<span style="color:#ff7777">✕ Generation failed.</span>'
        )

        with output:
            clear_output()
            print("GENERATION ERROR:")
            print(str(e))
            print("\nCheck ComfyUI log: /content/comfyui.log")

    finally:
        generate_button.disabled = False

generate_button.on_click(generate_image)

# =========================================================
# SHOW ALL FOUR KEYWORD LISTS
# =========================================================

keyword_summary = []

for filename, info in LORA_INFO.items():

    words = fetch_keywords(filename)

    keyword_summary.append(
        f"<b>{filename}</b><br>"
        + (
            ", ".join(words)
            if words
            else "<i>No published keywords returned / file not verified.</i>"
        )
    )

keyword_summary_html = (
    "<div class='studio-section'>"
    "<h3>📚 Four LoRA Trigger-Word Reference</h3>"
    + "<hr>".join(keyword_summary)
    + "</div>"
)

# =========================================================
# DISPLAY COMPLETE STUDIO
# =========================================================

display(widgets.VBox([

    section_title("01 — MODEL & LORA"),

    unet_dropdown,
    lora_dropdown,
    lora_strength,
    auto_keywords,
    keyword_status,
    keyword_display,
    refresh_button,

    widgets.HTML(keyword_summary_html),

    section_title("02 — PROMPT STUDIO"),

    prompt_box,
    negative_box,

    section_title("03 — IMAGE SETTINGS"),

    widgets.HBox([width_widget, height_widget]),
    steps_widget,
    cfg_widget,
    seed_widget,

    section_title("04 — ADVANCED SAMPLING"),

    sampler_widget,
    scheduler_widget,

    section_title("05 — GENERATE"),

    generate_button,
    status,
    output

], layout=widgets.Layout(width="100%")))

print("Z-Image Turbo Studio is ready.")
print("Available LoRAs:", len(lora_files))


Enter your Civitai API key to retrieve published trigger words.
Civitai API key (hidden, optional): ··········


Keyword lookup failed for Western_Woman_Boobs_1_Z-Image Turbo_000004500.safetensors: 404 Client Error: Not Found for url: https://civitai.com/api/v1/model-versions/2527219


Z-Image Turbo Studio is ready.
Available LoRAs: 4
